# EDA: Web Bot Detection Dataset

Разведочный анализ открытого датасета, на котором будем обучать базовую модель детектора ботов.
Графики и выводы собраны в отчёте [`reports/eda.md`](../reports/eda.md).

Данные: [Web Bot Detection Dataset](https://m4d.iti.gr/web-bot-detection-dataset/)
([Iliou et al., 2021](https://doi.org/10.1145/3447815)). Перед запуском их нужно скачать командой `make data`.
Ноутбук целиком выполняет `make eda`, графики сохраняются в `reports/figures`.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw" / "web_bot_detection_dataset"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

SESSION_GAP = pd.Timedelta(minutes=30)  # после такой паузы начинается новый визит
STROKE_GAP_MS = 200  # пауза, которая разделяет два движения мыши
MIN_STROKE_POINTS = 5
MIN_STROKE_LENGTH = 50  # в пикселях
EPOCH = pd.Timestamp("1970-01-01", tz="UTC")

PHASES = {"phase1": "Фаза 1", "phase2": "Фаза 2"}
CLASSES = {"human": "Люди", "moderate_bot": "Moderate-боты", "advanced_bot": "Advanced-боты"}
COLORS = {"human": "#2a78d6", "moderate_bot": "#eb6834", "advanced_bot": "#1baf7a"}
ACCENT, GRAYS = "#6250d6", ["#c3c2b7", "#e1e0d9"]
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
BLUES = LinearSegmentedColormap.from_list("blues", ["#eef5fe", "#86b6ef", "#2a78d6", "#0d366b"])

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "savefig.dpi": 150,
        "savefig.bbox": "tight",
        "font.size": 10,
        "text.color": INK,
        "axes.titlesize": 10.5,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.labelcolor": MUTED,
        "axes.edgecolor": "#c3c2b7",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "legend.frameon": False,
    }
)
pd.set_option("display.width", 200)
rng = np.random.default_rng(0)


def save(fig: plt.Figure, name: str) -> None:
    fig.savefig(FIGURES / name)
    plt.show()


def class_legend(fig: plt.Figure) -> None:
    handles = [
        Line2D([], [], marker="o", linestyle="", markersize=7, color=COLORS[label], label=name)
        for label, name in CLASSES.items()
    ]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 1.0), ncols=3)

## 1. Загрузка

Логи записаны в формате Apache, но вместо IP стоит прочерк, а перед User-Agent добавлен PHP session ID.

In [ ]:
LOG_RE = re.compile(r'^\S+ \S+ \[([^\]]+)\] "([^"]*)" (\d{3}) \S+ "([^"]*)" (\S+) "([^"]*)"\s*$')
LOG_COLUMNS = ["phase", "source", "ts", "request", "status", "referrer", "sid", "user_agent"]


def read_logs(data: Path) -> tuple[pd.DataFrame, int]:
    """Читает все логи. Возвращает запросы и число строк, которые не удалось разобрать."""
    rows, broken = [], 0
    for path in sorted(data.glob("phase*/data/web_logs/*/*.log")):
        phase, source = path.parts[-5], path.parts[-2]
        with path.open(encoding="utf-8", errors="replace") as file:
            for line in file:
                match = LOG_RE.match(line)
                if match is None:
                    broken += 1
                else:
                    rows.append((phase, source, *match.groups()))
    requests = pd.DataFrame(rows, columns=LOG_COLUMNS)
    requests["ts"] = pd.to_datetime(requests["ts"], format="%d/%b/%Y:%H:%M:%S %z")
    path = (
        requests["request"].str.split(" ").str[1].fillna("").str.replace(r"\?.*$", "", regex=True)
    )
    requests["path"] = path
    requests["kind"] = np.select(
        [
            path.str.startswith("/storage/"),  # сюда скрипт на странице отправляет движения мыши
            path.str.fullmatch(r"/|.*\.php"),
            path.str.contains(r"\.(?:css|js|ico|png|jpg|gif)$"),
        ],
        ["beacon", "page", "static"],
        default="other",
    )
    return requests, broken


def read_labels(data: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(data.glob("phase*/annotations/*/*")):
        lines = [line.split() for line in path.read_text().splitlines() if line.strip()]
        rows += [(path.parts[-4], path.parent.name, path.name, *line) for line in lines]
    return pd.DataFrame(rows, columns=["phase", "subset", "file", "session", "label"])


requests, broken_lines = read_logs(DATA)
labels_raw = read_labels(DATA)
labels = labels_raw.drop_duplicates(["phase", "session"])[["phase", "session", "label"]]
conflicts = labels_raw.groupby(["phase", "session"])["label"].nunique().gt(1).sum()

print(f"Строк лога разобрано: {len(requests):,}, не разобрано: {broken_lines}")
print(f"Размеченных сессий: {len(labels)}, сессий с противоречивой меткой: {conflicts}")
pd.crosstab(labels["phase"], labels["label"], margins=True)

В фазе 2 у меток есть суффикс `_0`, `_1`, `_2`: один PHP session ID может покрывать несколько визитов.
Правило разбиения в README не описано. Режем по паузе в 30 минут и проверяем, что так находятся все размеченные сессии.

In [ ]:
requests = requests.sort_values(["phase", "sid", "ts"], kind="stable").reset_index(drop=True)
new_visit = requests.groupby(["phase", "sid"])["ts"].diff() > SESSION_GAP
requests["visit"] = new_visit.groupby([requests["phase"], requests["sid"]]).cumsum()
requests["session"] = requests["sid"].where(
    requests["phase"] == "phase1", requests["sid"] + "_" + requests["visit"].astype(str)
)
requests = requests.merge(labels, on=["phase", "session"], how="left")
requests["ts_ms"] = (requests["ts"] - EPOCH) // pd.Timedelta(milliseconds=1)

found = labels.merge(requests[["phase", "session"]].drop_duplicates(), on=["phase", "session"])
long_gaps = (
    requests[requests["phase"] == "phase1"]
    .dropna(subset="label")
    .groupby("sid")["visit"]
    .max()
    .gt(0)
    .sum()
)
print(f"Размеченных сессий найдено в логах: {len(found)} из {len(labels)}")
print(f"Размеченных сессий фазы 1 с паузой больше 30 минут: {long_gaps}")

## 2. Объём и состав

In [ ]:
counts = pd.crosstab(labels["phase"], labels["label"])[list(CLASSES)]

fig, ax = plt.subplots(figsize=(6.4, 3.4))
width = 0.22
for shift, (label, name) in enumerate(CLASSES.items()):
    x = np.arange(len(PHASES)) + (shift - 1) * (width + 0.02)
    bars = ax.bar(x, counts[label], width, color=COLORS[label], label=name)
    ax.bar_label(bars, padding=2, color=INK)
ax.set_xticks(range(len(PHASES)), list(PHASES.values()))
ax.set_ylabel("Сессий")
ax.set_ylim(0, 68)
ax.grid(axis="x", visible=False)
ax.set_title("Размеченные сессии по классам", pad=28)
class_legend(fig)
save(fig, "01_classes.png")

In [ ]:
markup_names = {
    "labeled": "Размеченные сессии",
    "unlabeled": "Сессии без метки",
    "no_sid": "Без session ID",
}
requests["markup"] = np.select(
    [requests["label"].notna(), requests["sid"] == "-"], ["labeled", "no_sid"], default="unlabeled"
)
labeled = requests[requests["markup"] == "labeled"]
kind_names = {
    "beacon": "Отправка движений мыши",
    "page": "Страницы",
    "static": "Статика (css, js, иконки)",
}

by_markup = pd.crosstab(requests["phase"], requests["markup"])[list(markup_names)]
by_kind = labeled["kind"].value_counts()[list(kind_names)]
unlabeled_sessions = (
    requests[requests["markup"] == "unlabeled"].groupby("phase")["session"].nunique()
)
print(by_markup.assign(total=by_markup.sum(axis=1)))
print("Сессий без метки:", unlabeled_sessions.to_dict())
print(by_kind, "\nКоды ответа в размеченных сессиях:", labeled["status"].value_counts().to_dict())
pd.crosstab([labeled["phase"], labeled["label"]], labeled["kind"], margins=True)

In [ ]:
bars_data = {
    "Фаза 1, все строки лога": by_markup.loc["phase1"].rename(markup_names),
    "Фаза 2, все строки лога": by_markup.loc["phase2"].rename(markup_names),
    "Запросы размеченных сессий": by_kind.rename(kind_names),
}

fig, axes = plt.subplots(3, 1, figsize=(8.6, 3.9), gridspec_kw={"hspace": 1.5})
for ax, (title, values) in zip(axes, bars_data.items(), strict=True):
    shares = values / values.sum()
    left, small = 0.0, []
    for (name, share), color in zip(shares.items(), [ACCENT, *GRAYS], strict=True):
        ax.barh(0, share - 0.004, left=left + 0.002, height=0.6, color=color)
        if share > 0.2:
            text_color = "white" if color == ACCENT else INK
            ax.text(
                left + share / 2,
                0,
                f"{name}: {share:.0%}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )
        else:
            small.append(f"{name[0].lower()}{name[1:]}: {share:.1%}")
        left += share
    ax.text(1, -0.62, ", ".join(small), ha="right", va="center", fontsize=9)
    total = f"{values.sum():,}".replace(",", " ")
    ax.set_title(f"{title}: {total}", fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.9, 0.4)
    ax.axis("off")
save(fig, "02_log_composition.png")

## 3. Качество

In [ ]:
no_sid = requests[requests["markup"] == "no_sid"]
duplicates = labeled[labeled.duplicated(["phase", "session", "ts", "request"])]
endings = {
    str(path.relative_to(DATA / path.parts[-4])): "CRLF" if b"\r\n" in path.read_bytes() else "LF"
    for path in sorted(DATA.glob("phase*/annotations/*/*"))
}
print(f"Запросов без session ID: {len(no_sid):,} ({len(no_sid) / len(requests):.1%})")
print(no_sid["request"].str.slice(0, 40).value_counts().head(8))
print(
    f"Полных дублей в размеченных сессиях: {len(duplicates):,}, из них отправка мыши: {(duplicates['kind'] == 'beacon').sum():,}"
)
print("Переводы строк в файлах меток:", Counter(endings.values()))

## 4. Сессии по логам

Паузы считаем только между страницами. Отправка движений мыши идёт несколько раз в секунду и забивает все остальные запросы.

In [ ]:
KEYS = ["phase", "session"]
sessions = (
    labeled.assign(host=labeled["referrer"].str.extract(r"^(https?://[^/]+)")[0])
    .groupby(KEYS)
    .agg(
        label=("label", "first"),
        start=("ts", "min"),
        end=("ts", "max"),
        start_ms=("ts_ms", "min"),
        end_ms=("ts_ms", "max"),
        user_agent=("user_agent", "first"),
        host=("host", lambda values: values.mode().iloc[0]),
    )
)
sessions = sessions.join(pd.crosstab([labeled["phase"], labeled["session"]], labeled["kind"]))
sessions["duration_min"] = (sessions["end_ms"] - sessions["start_ms"]) / 60_000

pages = labeled[labeled["kind"] == "page"].copy()
pages["gap"] = pages.groupby(KEYS)["ts"].diff().dt.total_seconds()
sessions = sessions.join(
    pages.groupby(KEYS).agg(
        unique_pages=("path", "nunique"),
        gap_median=("gap", "median"),
        gap_mean=("gap", "mean"),
        gap_std=("gap", "std"),
    )
)
sessions["gap_cv"] = sessions["gap_std"] / sessions["gap_mean"]

LOG_FEATURES = {
    "duration_min": "Длительность сессии, мин",
    "page": "Просмотров страниц",
    "gap_median": "Медиана паузы между страницами, с",
}
sessions.groupby(["phase", "label"])[
    ["page", "unique_pages", "duration_min", "gap_median", "gap_cv"]
].agg(["min", "median", "max"]).round(1)

In [ ]:
def strip_panel(ax: plt.Axes, data: pd.DataFrame, column: str, *, log: bool) -> None:
    """Точка это сессия, чёрная черта это медиана класса."""
    for row, label in enumerate(CLASSES):
        values = data.loc[data["label"] == label, column].dropna()
        jitter = rng.uniform(-0.24, 0.24, len(values))
        ax.scatter(values, row + jitter, s=13, color=COLORS[label], alpha=0.5, linewidths=0)
        median = values.median()
        ax.plot([median, median], [row - 0.38, row + 0.38], color=INK, linewidth=2)
        ax.annotate(
            f"{median:.3g}",
            (median, row - 0.38),
            (0, 2),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
        )
    if log:
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.set_yticks(range(len(CLASSES)), list(CLASSES.values()))
    ax.set_ylim(len(CLASSES) - 0.45, -0.75)
    ax.grid(axis="y", visible=False)


def strip_figure(data: pd.DataFrame, features: dict[str, str], log: set[str], name: str) -> None:
    fig, axes = plt.subplots(
        2, len(features), figsize=(3.4 * len(features), 4.6), sharex="col", sharey=True
    )
    for row, (phase, phase_name) in enumerate(PHASES.items()):
        for col, (column, title) in enumerate(features.items()):
            ax = axes[row, col]
            strip_panel(ax, data.xs(phase, level="phase"), column, log=column in log)
            if row == 0:
                ax.set_title(title)
            ax.tick_params(axis="x", labelbottom=True)
        axes[row, 0].set_ylabel(phase_name, color=INK, fontweight="bold")
    fig.subplots_adjust(top=0.84, wspace=0.08, hspace=0.28)
    class_legend(fig)
    save(fig, name)


strip_figure(sessions, LOG_FEATURES, {"duration_min", "gap_median"}, "03_log_features.png")

## 5. Движения мыши

В фазе 1 на каждую сессию лежит отдельный JSON. В фазе 2 это выгрузка из MongoDB: одна запись на PHP session ID,
поэтому события раскладываем по визитам по времени. У кликов в фазе 2 время обрезано до девяти цифр, старшие разряды
берём у соседних событий.

In [ ]:
MOVE_RE = re.compile(r"m\((-?\d+),(-?\d+)\)")


def cv(values: np.ndarray) -> float:
    return float(np.std(values) / np.mean(values))


def trajectory_features(t: np.ndarray, x: np.ndarray, y: np.ndarray) -> dict[str, float]:
    """Признаки траектории. t в миллисекундах, x и y в пикселях, только события движения."""
    unordered = int((np.diff(t) < 0).sum())
    order = np.argsort(t, kind="stable")
    t, x, y = t[order], x[order], y[order]
    dt, step = np.diff(t), np.hypot(np.diff(x), np.diff(y))
    moving = (dt > 0) & (dt <= STROKE_GAP_MS)
    speed = step[moving] / dt[moving] * 1000
    bounds = np.flatnonzero(dt > STROKE_GAP_MS) + 1
    straightness = []
    for start, end in zip(np.r_[0, bounds], np.r_[bounds, len(t)], strict=True):
        length = step[start : end - 1].sum()
        if end - start >= MIN_STROKE_POINTS and length >= MIN_STROKE_LENGTH:
            straightness.append(np.hypot(x[end - 1] - x[start], y[end - 1] - y[start]) / length)
    return {
        "mouse_moves": len(t),
        "unordered": unordered,
        "strokes": len(straightness),
        "speed_median": float(np.median(speed)),
        "speed_cv": cv(speed),
        "event_dt_cv": cv(dt[moving]),
        "pause_median": float(np.median(dt[dt > STROKE_GAP_MS])) / 1000 if len(bounds) else np.nan,
        "straightness_median": float(np.median(straightness)) if straightness else np.nan,
    }


def read_mouse_phase1(data: Path) -> dict[tuple[str, str], dict[str, float]]:
    rows = {}
    for path in sorted(data.glob("phase1/data/mouse_movements/*/*/mouse_movements.json")):
        record = json.loads(path.read_text())
        times = np.array([int(value) for value in record["mousemove_times"].split(",") if value])
        xy = np.array(
            re.findall(r"\[(-?\d+),(-?\d+)\]", record["mousemove_total_behaviour"]), dtype=float
        )
        features = trajectory_features(times, xy[:, 0], xy[:, 1])
        features["clicks"] = record["total_behaviour"].count("[c(")
        rows["phase1", record["session_id"]] = features
    return rows


def read_mouse_phase2(data: Path, visits: pd.DataFrame) -> dict[tuple[str, str], dict[str, float]]:
    """visits: начало и конец каждого визита по логам в миллисекундах, индекс (sid, visit)."""
    rows = {}
    known = set(visits.index.get_level_values("sid"))
    lines = (
        line
        for path in sorted(data.glob("phase2/data/mouse_movements/*/*.json"))
        for line in path.open()
    )
    for line in lines:
        record = json.loads(line)
        sid = record["session_id"]
        if sid not in known:
            continue
        tokens = np.array(record["mousemove_total_behaviour"].strip("[]").split("]["))
        sizes = np.array(record["mousemove_client_height_width"].strip("[]").split("]["))
        urls = np.array(record["mousemove_visited_urls"].strip("[]").split("]["))
        times = pd.Series([int(value) for value in record["mousemove_times"].split(",") if value])
        truncated = (times < 10**12).to_numpy()
        prefix = (times // 10**9).where(~truncated).ffill().bfill()
        times = (prefix * 10**9 + times % 10**9).to_numpy().astype(np.int64)

        bounds = visits.loc[sid]
        cuts = (bounds["end_ms"].to_numpy()[:-1] + bounds["start_ms"].to_numpy()[1:]) / 2
        visit_of = np.searchsorted(cuts, times)
        is_move = np.char.startswith(tokens, "m(")
        for visit in np.unique(visit_of):
            in_visit = visit_of == visit
            moves = in_visit & is_move
            xy = np.array(MOVE_RE.findall("".join(tokens[moves])), dtype=float)
            features = trajectory_features(times[moves], xy[:, 0], xy[:, 1])
            features["clicks"] = int((in_visit & np.char.startswith(tokens, "c")).sum())
            features["truncated_times"] = int((in_visit & truncated).sum())
            features["window"] = Counter(sizes[in_visit]).most_common(1)[0][0]
            features["mouse_host"] = Counter(
                url.split("/")[2] for url in urls[in_visit]
            ).most_common(1)[0][0]
            rows["phase2", f"{sid}_{visit}"] = features
    return rows


visits = (
    requests[(requests["phase"] == "phase2") & requests["sid"].isin(labeled["sid"])]
    .groupby(["sid", "visit"])["ts_ms"]
    .agg(start_ms="min", end_ms="max")
)
mouse = pd.DataFrame.from_dict(
    {**read_mouse_phase1(DATA), **read_mouse_phase2(DATA, visits)}, orient="index"
)
mouse.index.names = KEYS
sessions = sessions.join(mouse)

print("Размеченных сессий без данных мыши:", sessions["mouse_moves"].isna().sum())
print(f"Событий движения мыши в размеченных сессиях: {sessions['mouse_moves'].sum():,.0f}")
print("Сессий, где события мыши идут не по порядку:", (sessions["unordered"] > 0).sum())
print(f"Кликов с обрезанным временем в фазе 2: {sessions['truncated_times'].sum():,.0f}")

MOUSE_FEATURES = {
    "speed_median": "Медиана скорости мыши, px/с",
    "speed_cv": "Разброс скорости мыши (CV)",
    "straightness_median": "Прямолинейность движений",
}
sessions.groupby(["phase", "label"])[
    [
        "mouse_moves",
        "clicks",
        "strokes",
        "speed_median",
        "speed_cv",
        "event_dt_cv",
        "pause_median",
        "straightness_median",
    ]
].median().round(2)

In [ ]:
strip_figure(sessions, MOUSE_FEATURES, {"speed_median"}, "04_mouse_features.png")

## 6. Утечки

Проверяем, чем классы различаются помимо поведения: временем сбора, браузером, адресом сайта и размером окна.

In [ ]:
info = sessions.reset_index()
info["day"] = info["start"].dt.strftime("%Y-%m-%d")
leaks = info.groupby(["phase", "label"]).agg(
    first_day=("day", "min"),
    last_day=("day", "max"),
    days=("day", "nunique"),
    user_agents=("user_agent", "nunique"),
    hosts=("host", lambda values: ", ".join(sorted(set(values)))),
    windows=("window", "nunique"),
)
print(leaks.to_string())
bot_days = set(info.loc[info["label"] != "human", "day"])
print(
    "Сессий людей в дни, когда собирали ботов:",
    info.loc[info["label"] == "human", "day"].isin(bot_days).sum(),
)
info["os"] = info["user_agent"].str.extract(r"\(([^;)]+)")[0].str.replace(r" NT.*", "", regex=True)
pd.crosstab([info["phase"], info["label"]], info["os"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True, gridspec_kw={"wspace": 0.06})
for ax, (phase, phase_name) in zip(axes, PHASES.items(), strict=True):
    part = sessions.xs(phase, level="phase")
    for row, label in enumerate(CLASSES):
        starts = part.loc[part["label"] == label, "start"]
        jitter = rng.uniform(-0.25, 0.25, len(starts))
        ax.scatter(starts, row + jitter, s=14, color=COLORS[label], alpha=0.55, linewidths=0)
    ax.set_title(f"{phase_name}: когда началась сессия")
    ax.set_yticks(range(len(CLASSES)), list(CLASSES.values()))
    ax.set_ylim(len(CLASSES) - 0.5, -0.5)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="x", labelsize=8.5)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%Y"))
axes[0].xaxis.set_major_locator(mdates.DayLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
fig.subplots_adjust(top=0.78)
class_legend(fig)
save(fig, "05_collection_time.png")

## 7. Насколько классы разделяет один признак

Для каждого признака считаем ROC AUC «бот против человека»: 0.5 значит, что признак ничего не даёт,
0 и 1 значат, что он разделяет классы полностью. На графике показано расстояние до 0.5 в обе стороны.

In [ ]:
FEATURES = {
    **LOG_FEATURES,
    "unique_pages": "Уникальных страниц",
    "gap_cv": "Разброс пауз между страницами (CV)",
    "clicks": "Кликов",
    "strokes": "Длинных движений мыши",
    **MOUSE_FEATURES,
    "event_dt_cv": "Разброс интервалов между событиями мыши (CV)",
    "pause_median": "Медиана паузы между движениями, с",
}
PAIRS = {
    (phase, bot): f"{PHASES[phase]}\n{CLASSES[bot].split('-')[0]}"
    for phase in PHASES
    for bot in ["moderate_bot", "advanced_bot"]
}


def auc(bots: pd.Series, humans: pd.Series) -> float:
    """Вероятность того, что у случайного бота признак больше, чем у случайного человека."""
    bots, humans = bots.dropna(), humans.dropna()
    ranks = pd.concat([bots, humans]).rank()
    return (ranks.iloc[: len(bots)].sum() - len(bots) * (len(bots) + 1) / 2) / (
        len(bots) * len(humans)
    )


def pair_auc(phase: str, bot: str) -> list[float]:
    part = sessions.xs(phase, level="phase")
    return [
        auc(part.loc[part["label"] == bot, f], part.loc[part["label"] == "human", f])
        for f in FEATURES
    ]


aucs = pd.DataFrame({pair: pair_auc(*pair) for pair in PAIRS}, index=list(FEATURES))
separability = aucs.sub(0.5).abs().add(0.5)
print("Худшая разделимость признака по четырём сравнениям:")
print(separability.min(axis=1).rename(FEATURES).round(3).to_string())
aucs.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5.2))
image = ax.imshow(separability, cmap=BLUES, vmin=0.5, vmax=1.0, aspect="auto")
for row in range(len(FEATURES)):
    for col in range(len(PAIRS)):
        value, raw = separability.iloc[row, col], aucs.iloc[row, col]
        mark = "▲" if raw > 0.5 else "▼"
        ax.text(
            col,
            row,
            f"{value:.2f} {mark}",
            ha="center",
            va="center",
            color="white" if value > 0.8 else INK,
            fontsize=9,
        )
ax.set_xticks(range(len(PAIRS)), list(PAIRS.values()))
ax.set_yticks(range(len(FEATURES)), list(FEATURES.values()))
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False, length=0)
ax.grid(visible=False)
ax.set_xticks(np.arange(-0.5, len(PAIRS)), minor=True)
ax.set_yticks(np.arange(-0.5, len(FEATURES)), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(image, ax=ax, fraction=0.035, pad=0.03, label="Разделимость (0.5 нет, 1.0 полная)")
fig.text(
    0.02, 0.0, "▲ у ботов значение больше, чем у людей. ▼ у ботов меньше.", color=MUTED, fontsize=9
)
save(fig, "06_separability.png")

Порог, подобранный на фазе 1, проверяем на фазе 2: так видно, какие признаки переносятся между фазами.

In [ ]:
def best_threshold(values: pd.Series, is_bot: pd.Series) -> tuple[float, bool]:
    """Порог с лучшей точностью и флаг: считать ботом значения выше порога или ниже."""
    candidates = np.sort(values.unique())
    candidates = (candidates[:-1] + candidates[1:]) / 2
    accuracy = np.array([((values > c) == is_bot).mean() for c in candidates])
    above = accuracy.max() >= (1 - accuracy).max()
    return float(candidates[accuracy.argmax() if above else accuracy.argmin()]), bool(above)


train, test = sessions.xs("phase1", level="phase"), sessions.xs("phase2", level="phase")
rows = []
for column, title in FEATURES.items():
    threshold, above = best_threshold(train[column], train["label"] != "human")
    predicted = (test[column] > threshold) == above
    is_bot = test["label"] != "human"
    rows.append(
        {
            "Признак": title,
            "Порог из фазы 1": round(threshold, 2),
            "Бот, если": "больше" if above else "меньше",
            "Точность на фазе 1": (
                ((train[column] > threshold) == above) == (train["label"] != "human")
            ).mean(),
            "Точность на фазе 2": (predicted == is_bot).mean(),
            "Recall ботов на фазе 2": predicted[is_bot].mean(),
            "FPR на фазе 2": predicted[~is_bot].mean(),
        }
    )
pd.DataFrame(rows).set_index("Признак").round(2)